# Centaur IME: ベトナム語と韓国語の並べ替えの模型を長く学習する（幅 256・4 層のまま）

大きさは今の小さい模型と同じ（IME の速さは変わらない）。学習の文を増やし、それぞれ 110 分学習する。
使い方: L4 GPU を選び、「すべてのセルを実行」。最初のセルで Google ドライブにつなぐ。最後にランタイムを切る。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/CentaurIME/models/2026-11-08'
import os; os.makedirs(OUT, exist_ok=True); print('保存先', OUT)

In [ ]:
!rm -rf repo && git clone -q --depth 1 -b work/lang-measure https://github.com/J-KAMMA/centaur-ime.git repo && git -C repo log -1 --format='%H %s'
!pip -q install pyarrow==21.0.0 pypinyin==0.53.0 "opencc>=1.1,<2"

In [ ]:
!python repo/tools/fetch_source.py tatoeba-vie data/vi
!python repo/tools/fetch_source.py wiki-vi data/vi
!python repo/tools/fetch_source.py cc100-vi data/vi --lines 2000000

In [ ]:
!python repo/tools/lang_measure/vi_measure.py --tatoeba data/vi/vie_sentences.tsv.bz2 --wiki $(ls data/vi/*.parquet | head -1) --out mvi --nbest-out nbest-vi.jsonl
!wc -l nbest-vi.jsonl

In [ ]:
!python repo/tools/models/train_task.py --task rr-vi --tatoeba data/vi/vie_sentences.tsv.bz2 --cc100 data/vi/cc100-vi.2000000.txt --extra 2000000 --d 256 --layers 4 --heads 4 --batch 256 --minutes 110 --amp --nbest-file nbest-vi.jsonl --out "$OUT" --name rr-vi-d256-long
print(open(OUT + '/rr-vi-d256-long-result.txt', encoding='utf-8').read())

In [ ]:
!python repo/tools/fetch_source.py tatoeba-kor data/ko
!python repo/tools/fetch_source.py wiki-ko data/ko
!python repo/tools/fetch_source.py cc100-ko data/ko --lines 1500000

In [ ]:
!python repo/tools/ko_ime/measure.py --tatoeba data/ko/kor_sentences.tsv.bz2 --wiki $(ls data/ko/*.parquet | head -1) --out mko --nbest-out nbest-ko.jsonl
!wc -l nbest-ko.jsonl

In [ ]:
!python repo/tools/models/train_task.py --task rr-ko --tatoeba data/ko/kor_sentences.tsv.bz2 --wiki $(ls data/ko/*.parquet | head -1) --cc100 data/ko/cc100-ko.1500000.txt --extra 1500000 --batch 256 --amp --nbest-file nbest-ko.jsonl --d 256 --layers 4 --heads 4 --minutes 110 --out "$OUT" --name rr-ko-d256-long
print(open(OUT + '/rr-ko-d256-long-result.txt', encoding='utf-8').read())

In [ ]:
from google.colab import runtime
runtime.unassign()